# Introduction



# Instruct Fine-Tuning LLMs on Snellius with Unsloth + QLoRA

This tutorial showcases how to **instruct-finetune a Large Language Model (LLM)** using **[Unsloth](https://github.com/unslothai/unsloth)**.  
It serves as supplementary material to the _LLM Finetune on Snellius_ code.

Here, we focus on **efficiently and effectively using Snellius resources**. To this end, we adopt **Unsloth's QLoRA implementation**.

> **QLoRA** significantly reduces memory requirements for fine-tuning without a notable loss in precision (Dettmers et al., 2023).  
> Lower memory usage allows us to train **larger models** or increase the **batch size**!

---

## Table of Contents

- [Introduction to QLoRA](#introduction-to-qlora)
- [Walkthrough](#walkthrough)
  - [Import necessities](#import-necessities)
  - [Setting up the finetune config](#setting-up-the-finetune-config)
  - [Loading the model and tokenizer](#loading-the-model-and-tokenizer)
  - [Loading the dataset](#loading-the-dataset)
  - [QLoRA magic](#qlora-magic)
  - [Start finetuning!](#start-finetuning)
- [Running on SLURM](#running-on-slurm)
- [Generation](#Generation)
- [Source and Further Reading](#source-and-further-reading)

---

## Introduction to QLoRA

**Quantized Low-Rank Adaptation (QLoRA)** keeps the base model weights **frozen**, and instead learns a **decomposable weight matrix** that is applied on top of the static foundation model.  
This strategy avoids computing and storing the **memory-intensive gradients and optimizer states** for the full model.

- The trainable weight matrix is controlled via the hyperparameter `r`, which defines the **rank** of the matrix.
- Typically, `r=16` is used, resulting in trainable weights that are only **~0.5% of the original model size**.
- The original QLoRA paper shows that performance is not very sensitive to `r`.

QLoRA **extends LoRA** by first **quantizing** the base model to **4-bit NormalFloat (NF4)** precision. This drastically reduces memory usage, allowing the model to fit on limited GPU memory.

> ⚠️ Even with quantization, the model still needs to be loaded onto the GPU for applying weight updates and computing the loss.

---

## Why Unsloth?

Unsloth is an optimized framework for instruction tuning using QLoRA. It includes:
- Smart optimizations (e.g. patched attention, fused operations)
- Easy integration with the HuggingFace ecosystem
- Efficient fine-tuning of models like LLaMA 2, Mistral, and others

---

> 💡 **Did you know?**  
> LoRA originated in **diffusion models** for image generation, such as **Stable Diffusion**, before becoming popular for LLM fine-tuning!


In [1]:
# Install Dependency (if needed)

# Install required libraries (run only once)
# !pip install unsloth trl transformers datasets accelerate ipywidgets

## Import Necessities

As always, start off by defining the imports.

All libraries are dependencies from Unsloth.
The environment variable DATA_PATH is also set where the model, tokenizer and dataset will be downloaded to!

In [2]:
import os
from datasets import load_dataset

from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template
from trl import SFTConfig, SFTTrainer


# DATA_PATH = os.getenv('TEACHER_DIR', os.getcwd()) + '/JHS_data'
DATA_PATH = "./"

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Setting up the finetune config

Next up is defining the supervised finetuning config. These are the necessary parameters we need to pass to the trainer.

The output directory will be used to store intermediate and the final checkpoint of the adapter model and some log information.

The maximum sequence length (max_seq_length) signifies the ability for the LLM to comprehend the information in a single pass. Thus, a sequence (or context) length of 8192 means the model can utilize its window span of 8192 tokens (roughly 10,000 words) to retrieve relevant information and return an answer.

In [3]:
max_seq_length = 8192
training_args = SFTConfig(
    dataset_text_field="text",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",
    max_steps=200,
    max_seq_length=max_seq_length,
    output_dir=f"{DATA_PATH}/qlora_finetune",
    dataset_num_proc=len(os.sched_getaffinity(0))

)

training_args.dataset_text_field = "text"


## Loading the model and tokenizer

Let's start by loading the model and tokenizer. As we use QLoRA, we want to load the model in 4-bit precision. Here, we load a pre-quantized 4-bit model by Unsloth to reduce the quantization time. Alternatively, you can load a different model (see here for all supported [Unsloth models](https://docs.unsloth.ai/get-started/all-our-models)
) even when it's not quantized like the official [Meta-Llama-3.1](https://huggingface.co/meta-llama/Llama-3.1-8B) from HuggingFace. Keep in mind that some of these models are 'gated' models and can only be downloaded after accepting terms and conditions. Please follow [this tutorial](https://huggingface.co/docs/hub/en/models-gated) if you encountered a 'gated' model or dataset.

In [4]:
model, tokenizer = FastLanguageModel.from_pretrained(
    # model_name="unsloth/Llama-3.2-1B-unsloth-bnb-4bit", #bigger model: unsloth/Meta-Llama-3.1-8B-bnb-4bit OR unsloth/Llama-3.2-3B-bnb-4bit
    model_name="unsloth/Qwen2.5-0.5B-unsloth-bnb-4bit", # If you get OOM
    max_seq_length=max_seq_length,
    device_map="auto",
    dtype=None,  # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.3.3: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA A100-SXM4-40GB MIG 1g.5gb. Num GPUs = 1. Max memory: 4.75 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

unsloth/Qwen2.5-0.5B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


## Loading the dataset

As a dataset, we opted for [Open-Orca/SlimOrca](https://huggingface.co/datasets/Open-Orca/SlimOrca) due to its standard formatting, easy for integration into these frameworks and applicability for targeted instruction learning. 

Because there is no explicit chat template defined for foundation models, we choose for the "chatml" template which is one the standards nowadays. In case you are finetuning from an existing instruct-finetuned LLM, then skip the foundation_model flag and avoid getting the chat template.

Lastly, a mapping is performed to convert the nested dictionary samples into a list of formatted strings with their corresponding chat template. An example to illustrate:

```python
{ "from": "system", "value": "You are an AI assistant. Provide a detailed answer so user don’t need to search outside to understand the answer.", "weight": null }, 
{ "from": "human", "value": "Please answer the following question: - They are buried under layers of soil - Pressure builds over time - The remains liquefy - The carbon atoms rearrange to become a new substance. What might be the first step of the process?\nA:", "weight": 0 }, 
{ "from": "gpt", "value": "A: The first step of the process is \"They are buried under layers of soil.\" This occurs when the remains of plants, animals, or other organic material become covered by soil and other sediments. Over time, as more and more layers accumulate, the pressure and heat increase, eventually leading to the transformation of the remains into substances like coal, oil, or natural gas.", "weight": 1 } ]
```
is converted to a single string of:

```
system: You are an AI assistant. Provide a detailed answer so user don’t need to search outside to understand the answer.

user: Please answer the following question: - They are buried under layers of soil - Pressure builds over time - The remains liquefy - The carbon atoms rearrange to become a new substance. What might be the first step of the process?

assistant", "value": "A: The first step of the process is \"They are buried under layers of soil.\" This occurs when the remains of plants, animals, or other organic material become covered by soil and other sediments. Over time, as more and more layers accumulate, the pressure and heat increase, eventually leading to the transformation of the remains into substances like coal, oil, or natural gas.
```

Keep in mind that especially this step is subject to change for your dataset!

In [5]:
dataset = load_dataset("Open-Orca/SlimOrca", split="train")
# Define own template if finetuning from pre-trained model. If continue from a instruct finetune, then use the native tokenizer and chat template
chat_tokenizer = get_chat_template(
	tokenizer,
	mapping={
	"role": "from",
	"content": "value",
	"user": "human",
	"assistant": "gpt",
	},
	chat_template="chatml",
	map_eos_token=True,
)

# Dataset-specific function to convert the samples (in dictionaries) to strings with corresponding template
def formatting_prompts_func(examples):
	convos = examples["conversations"]
	texts = [
		chat_tokenizer.apply_chat_template(
		convo, tokenize=False, add_generation_prompt=False
		)
		for convo in convos
	]
	return {
		"text": texts,
	}

# Apply applying the samples dictionaries to string
dataset = dataset.map(
	formatting_prompts_func, batched=True, num_proc=len(os.sched_getaffinity(0))
	)

Unsloth: Will map <|im_end|> to EOS = <|endoftext|>.


## QLoRA magic

Below, the QLoRA is applied to the original model. The QLoRA hyperparameters are set here. Generally, these settings work well already. If you want to know more, we recommend reading this [blog post](https://medium.com/@dillipprasad60/qlora-explained-a-deep-dive-into-parametric-efficient-fine-tuning-in-large-language-models-llms-c1a4794b1766).

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # rank of parameters. Higher R means more parameters
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,  # scaling of the weights
    lora_dropout=0,  # Dropout = 0 is currently optimized
    bias="none",  # Bias = "none" is currently optimized
    use_gradient_checkpointing="unsloth",
    max_seq_length=max_seq_length,
    random_state=47,
)

Unsloth 2026.3.3 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


## Start finetuning!

Lastly, initialize the trainer and already start training. It's that easy!

After training, the static model and the weight matrix, or adapter, is saved at the output directory in your /scratch-shared/ folder. An adapter can then be merged with the original foundation model to obtain the final model.

In [7]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=dataset,
)
os.environ['UNSLOTH_RETURN_LOGITS'] = '1'
trainer_stats = trainer.train()
print(trainer_stats)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/517982 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 517,982 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss
1,2.050440
2,1.921340
3,1.908807
4,1.603963
5,2.516968
6,2.507262
7,1.901396
8,2.170791
9,1.762624
10,1.780205


TrainOutput(global_step=200, training_loss=1.69321487814188, metrics={'train_runtime': 173.7449, 'train_samples_per_second': 4.604, 'train_steps_per_second': 1.151, 'total_flos': 698208861488640.0, 'train_loss': 1.69321487814188, 'epoch': 0.0015444552127293998})


## Running on SLURM

See [here](https://github.com/sara-nl/LLM-finetune) for the complete runnable codebase including job script



## Generation

In this part, you **interactively chat** with your fine-tuned model using `Unsloth`'s `FastLanguageModel`.


In [8]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from unsloth import FastLanguageModel

def jupyter_chat(model, tokenizer, max_new_tokens=128, temperature=0.8):
    FastLanguageModel.for_inference(model)

    history = []

    input_box = widgets.Textarea(
        placeholder='Type your message here...',
    )

    output_box = widgets.Output()
    send_button = widgets.Button(description="Send")
    def on_send_clicked(b):
        user_input = input_box.value.strip()
        input_box.value = ""

        if user_input.lower() == "exit":
            with output_box:
                print("Goodbye!")
            return

        elif user_input.lower() == "reset":
            history.clear()
            with output_box:
                clear_output()
                print("Chat history cleared.")
            return

        history.append(f"<|user|>\n{user_input}\n<|assistant|>")
        prompt = "\n".join(history)

        inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[1]

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )

        response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
        history.append(response)

        with output_box:
            print(f"\nYou: {user_input}\nModel: {response}\n")

    send_button.on_click(on_send_clicked)

    display(input_box, send_button, output_box)

In [23]:
jupyter_chat(model, tokenizer, temperature=0.8)

Textarea(value='', placeholder='Type your message here...')

Button(description='Send', style=ButtonStyle())

Output()

## Source and further reading
- LoRA paper: https://arxiv.org/abs/2106.09685
- QLoRA paper: https://arxiv.org/abs/2305.14314
- SFTTrainer documentation: https://huggingface.co/docs/trl/main/en/sft_trainer
- Excellent LLM notebooks: https://github.com/mlabonne/llm-course